In [ ]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from tqdm import tqdm
import pandas as pd

# config (edit as needed)
MODEL = "Qwen/Qwen3-14B"
SYSTEM_PROMPT = "You are an expert at generating realistic and culturally-relevant math word problems tailored to the country."
INPUT_PATH = "data/incontext_albanian.csv"
OUTPUT_PATH_1 = "outputs/incontext_albanian_qwen.xlsx"
OUTPUT_PATH_2 = "outputs/incontext_albanian_qwen_extracted.xlsx"
BATCH_SIZE = 32

# load data, model, tokenizer
df = pd.read_csv(INPUT_PATH)
llm = LLM(model=MODEL, tensor_parallel_size=1, download_dir="/nesi/nobackup/massey04342/models", gpu_memory_utilization=0.9)
# For thinking mode (enable_thinking=True), use Temperature=0.6, TopP=0.95, TopK=20, and MinP=0. 
#DO NOT use greedy decoding, as it can lead to performance degradation and endless repetitions.
sampling = SamplingParams(temperature=0.6, top_p=0.95, top_k=20, max_tokens=5120)
tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True, cache_dir="/nesi/nobackup/massey04342/models")

# build prompts (None for empties)
prompts = []
for p in df["zeroshot_prompt"]:
    if pd.isna(p) or str(p).strip() == "":
        prompts.append(None)
    else:
        prompts.append(f"<|system|>\n{SYSTEM_PROMPT}\n<|user|>\n{p}\n<|assistant|>\n")

# indices & results placeholder
non_empty_indices = [i for i, p in enumerate(prompts) if p is not None]
results = [""] * len(prompts)

# batch loop (simple)
for start in tqdm(range(0, len(non_empty_indices), BATCH_SIZE), desc="Batched generation"):
    batch_indices = non_empty_indices[start:start + BATCH_SIZE]
    batch_prompts = [prompts[i] for i in batch_indices]
    try:
        outs = llm.generate(batch_prompts, sampling_params=sampling)
        for j, out in enumerate(outs):
            text = tokenizer.decode(out.outputs[0].token_ids, skip_special_tokens=True).strip("\n")
            results[batch_indices[j]] = text
    except Exception as e:
        for i in batch_indices:
            results[i] = f"[ERROR] {type(e).__name__}: {str(e)[:200]}"

# save back to dataframe
df["zeroshot_response"] = results




prompts = []
for p in df["oneshot_prompt"]:
    if pd.isna(p) or str(p).strip() == "":
        prompts.append(None)
    else:
        messages = [
            {
                "role": "system",
                "content": [{"type": "text", "text": SYSTEM_PROMPT}]
            },
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": p}
                ]
            }
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        prompts.append(prompt)

    
# indices & results placeholder
non_empty_indices = [i for i, p in enumerate(prompts) if p is not None]
results = [""] * len(prompts)


# batch loop (simple)
for start in tqdm(range(0, len(non_empty_indices), BATCH_SIZE), desc="Batched generation"):
    batch_indices = non_empty_indices[start:start + BATCH_SIZE]
    batch_prompts = [prompts[i] for i in batch_indices]
    try:
        outs = llm.generate(batch_prompts, sampling_params=sampling)
        for j, out in enumerate(outs):
            text = tokenizer.decode(out.outputs[0].token_ids, skip_special_tokens=True).strip("\n")
            results[batch_indices[j]] = text
    except Exception as e:
        for i in batch_indices:
            results[i] = f"[ERROR] {type(e).__name__}: {str(e)[:200]}"

# save back to dataframe
df["oneshot_response"] = results



df.to_excel(OUTPUT_PATH_1, index=False, engine="openpyxl")

(EngineCore_DP0 pid=3634831) 

INFO 05-01 13:21:45 [gpu_worker.py:375] Available KV cache memory: 50.41 GiB


(EngineCore_DP0 pid=3634831) 

INFO 05-01 13:21:45 [kv_cache_utils.py:1291] GPU KV cache size: 330,352 tokens


(EngineCore_DP0 pid=3634831) 

...

Batched generation:  90%|█████████ | 9/10 [09:18<00:59, 59.89s/it]

Adding requests:   0%|          | 0/4 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/4 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Batched generation: 100%|██████████| 10/10 [09:33<00:00, 46.06s/it]

Batched generation: 100%|██████████| 10/10 [09:33<00:00, 57.32s/it]

In [3]:
import json
import re
import ast

def _extract_last_json_str(s):
    if not isinstance(s, str):
        return None
    i = s.rfind("{")
    if i == -1:
        return None
    # walk forward to find matching closing brace (handles nested braces)
    depth = 0
    end = None
    for j in range(i, len(s)):
        c = s[j]
        if c == "{":
            depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0:
                end = j + 1
                break
    candidate = s[i:end] if end is not None else s[i:]  # if no closing brace, take to end
    return candidate.strip()

def _parse_loose_json(candidate):
    if candidate is None:
        return None
    # 1) Try strict JSON
    try:
        return json.loads(candidate)
    except Exception:
        pass
    # 2) Quick heuristics: single->double quotes, remove trailing commas before } or ]
    cand = candidate.replace("'", '"')
    cand = re.sub(r",\s*([}\]])", r"\1", cand)
    try:
        return json.loads(cand)
    except Exception:
        pass
    # 3) ast.literal_eval as a last structured attempt (can handle Python dicts)
    try:
        return ast.literal_eval(candidate)
    except Exception:
        pass
    # 4) Give up and return the raw extracted string
    return candidate

# Apply to dataframe (no in-place overwrite until all succeed)
df["extracted_zeroshot_response"] = df["zeroshot_response"].apply(lambda s: _parse_loose_json(_extract_last_json_str(s)))
df["extracted_oneshot_response"] = df["oneshot_response"].apply(lambda s: _parse_loose_json(_extract_last_json_str(s)))
df.to_excel(OUTPUT_PATH_2, index=False, engine="openpyxl")

In [ ]:


# config (edit as needed)
INPUT_PATH = "data/incontext_hindi.csv"
OUTPUT_PATH_1 = "outputs/incontext_hindi_qwen.xlsx"
OUTPUT_PATH_2 = "outputs/incontext_hindi_qwen_extracted.xlsx"
BATCH_SIZE = 32


# load data, model, tokenizer
df = pd.read_csv(INPUT_PATH)

# build prompts (None for empties)
prompts = []
for p in df["zeroshot_prompt"]:
    if pd.isna(p) or str(p).strip() == "":
        prompts.append(None)
    else:
        messages = [
            {
                "role": "system",
                "content": [{"type": "text", "text": SYSTEM_PROMPT}]
            },
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": p}
                ]
            }
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        prompts.append(prompt)

    
# indices & results placeholder
non_empty_indices = [i for i, p in enumerate(prompts) if p is not None]
results = [""] * len(prompts)

# batch loop (simple)
for start in tqdm(range(0, len(non_empty_indices), BATCH_SIZE), desc="Batched generation"):
    batch_indices = non_empty_indices[start:start + BATCH_SIZE]
    batch_prompts = [prompts[i] for i in batch_indices]
    try:
        outs = llm.generate(batch_prompts, sampling_params=sampling)
        for j, out in enumerate(outs):
            text = tokenizer.decode(out.outputs[0].token_ids, skip_special_tokens=True).strip("\n")
            results[batch_indices[j]] = text
    except Exception as e:
        for i in batch_indices:
            results[i] = f"[ERROR] {type(e).__name__}: {str(e)[:200]}"

# save back to dataframe
df["zeroshot_response"] = results




prompts = []
for p in df["oneshot_prompt"]:
    if pd.isna(p) or str(p).strip() == "":
        prompts.append(None)
    else:
        messages = [
            {
                "role": "system",
                "content": [{"type": "text", "text": SYSTEM_PROMPT}]
            },
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": p}
                ]
            }
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        prompts.append(prompt)

    
# indices & results placeholder
non_empty_indices = [i for i, p in enumerate(prompts) if p is not None]
results = [""] * len(prompts)

# batch loop (simple)
for start in tqdm(range(0, len(non_empty_indices), BATCH_SIZE), desc="Batched generation"):
    batch_indices = non_empty_indices[start:start + BATCH_SIZE]
    batch_prompts = [prompts[i] for i in batch_indices]
    try:
        outs = llm.generate(batch_prompts, sampling_params=sampling)
        for j, out in enumerate(outs):
            text = tokenizer.decode(out.outputs[0].token_ids, skip_special_tokens=True).strip("\n")
            results[batch_indices[j]] = text
    except Exception as e:
        for i in batch_indices:
            results[i] = f"[ERROR] {type(e).__name__}: {str(e)[:200]}"

# save back to dataframe
df["oneshot_response"] = results


df.to_excel(OUTPUT_PATH_1, index=False, engine="openpyxl")

In [ ]:
# Apply to dataframe (no in-place overwrite until all succeed)
df["extracted_zeroshot_response"] = df["zeroshot_response"].apply(lambda s: _parse_loose_json(_extract_last_json_str(s)))
df["extracted_oneshot_response"] = df["oneshot_response"].apply(lambda s: _parse_loose_json(_extract_last_json_str(s)))
df.to_excel(OUTPUT_PATH_2, index=False, engine="openpyxl")

In [ ]:


# config (edit as needed)
INPUT_PATH = "data/incontext_punjabi.csv"
OUTPUT_PATH_1 = "outputs/incontext_punjabi_qwen.xlsx"
OUTPUT_PATH_2 = "outputs/incontext_punjabi_qwen_extracted.xlsx"
BATCH_SIZE = 32


# load data, model, tokenizer
df = pd.read_csv(INPUT_PATH)

# build prompts (None for empties)
prompts = []
for p in df["zeroshot_prompt"]:
    if pd.isna(p) or str(p).strip() == "":
        prompts.append(None)
    else:
        messages = [
            {
                "role": "system",
                "content": [{"type": "text", "text": SYSTEM_PROMPT}]
            },
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": p}
                ]
            }
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        prompts.append(prompt)

    
# indices & results placeholder
non_empty_indices = [i for i, p in enumerate(prompts) if p is not None]
results = [""] * len(prompts)

# batch loop (simple)
for start in tqdm(range(0, len(non_empty_indices), BATCH_SIZE), desc="Batched generation"):
    batch_indices = non_empty_indices[start:start + BATCH_SIZE]
    batch_prompts = [prompts[i] for i in batch_indices]
    try:
        outs = llm.generate(batch_prompts, sampling_params=sampling)
        for j, out in enumerate(outs):
            text = tokenizer.decode(out.outputs[0].token_ids, skip_special_tokens=True).strip("\n")
            results[batch_indices[j]] = text
    except Exception as e:
        for i in batch_indices:
            results[i] = f"[ERROR] {type(e).__name__}: {str(e)[:200]}"

# save back to dataframe
df["zeroshot_response"] = results




prompts = []
for p in df["oneshot_prompt"]:
    if pd.isna(p) or str(p).strip() == "":
        prompts.append(None)
    else:
        messages = [
            {
                "role": "system",
                "content": [{"type": "text", "text": SYSTEM_PROMPT}]
            },
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": p}
                ]
            }
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        prompts.append(prompt)

    
# indices & results placeholder
non_empty_indices = [i for i, p in enumerate(prompts) if p is not None]
results = [""] * len(prompts)

# batch loop (simple)
for start in tqdm(range(0, len(non_empty_indices), BATCH_SIZE), desc="Batched generation"):
    batch_indices = non_empty_indices[start:start + BATCH_SIZE]
    batch_prompts = [prompts[i] for i in batch_indices]
    try:
        outs = llm.generate(batch_prompts, sampling_params=sampling)
        for j, out in enumerate(outs):
            text = tokenizer.decode(out.outputs[0].token_ids, skip_special_tokens=True).strip("\n")
            results[batch_indices[j]] = text
    except Exception as e:
        for i in batch_indices:
            results[i] = f"[ERROR] {type(e).__name__}: {str(e)[:200]}"

# save back to dataframe
df["oneshot_response"] = results


df.to_excel(OUTPUT_PATH_1, index=False, engine="openpyxl")

In [ ]:

# Apply to dataframe (no in-place overwrite until all succeed)
df["extracted_zeroshot_response"] = df["zeroshot_response"].apply(lambda s: _parse_loose_json(_extract_last_json_str(s)))
df["extracted_oneshot_response"] = df["oneshot_response"].apply(lambda s: _parse_loose_json(_extract_last_json_str(s)))
df.to_excel(OUTPUT_PATH_2, index=False, engine="openpyxl")

In [ ]:


# config (edit as needed)
INPUT_PATH = "data/incontext_odia.csv"
OUTPUT_PATH_1 = "outputs/incontext_odia_qwen.xlsx"
OUTPUT_PATH_2 = "outputs/incontext_odia_qwen_extracted.xlsx"
BATCH_SIZE = 32


# load data, model, tokenizer
df = pd.read_csv(INPUT_PATH)

# build prompts (None for empties)
prompts = []
for p in df["zeroshot_prompt"]:
    if pd.isna(p) or str(p).strip() == "":
        prompts.append(None)
    else:
        messages = [
            {
                "role": "system",
                "content": [{"type": "text", "text": SYSTEM_PROMPT}]
            },
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": p}
                ]
            }
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        prompts.append(prompt)

    
# indices & results placeholder
non_empty_indices = [i for i, p in enumerate(prompts) if p is not None]
results = [""] * len(prompts)

# batch loop (simple)
for start in tqdm(range(0, len(non_empty_indices), BATCH_SIZE), desc="Batched generation"):
    batch_indices = non_empty_indices[start:start + BATCH_SIZE]
    batch_prompts = [prompts[i] for i in batch_indices]
    try:
        outs = llm.generate(batch_prompts, sampling_params=sampling)
        for j, out in enumerate(outs):
            text = tokenizer.decode(out.outputs[0].token_ids, skip_special_tokens=True).strip("\n")
            results[batch_indices[j]] = text
    except Exception as e:
        for i in batch_indices:
            results[i] = f"[ERROR] {type(e).__name__}: {str(e)[:200]}"

# save back to dataframe
df["zeroshot_response"] = results




prompts = []
for p in df["oneshot_prompt"]:
    if pd.isna(p) or str(p).strip() == "":
        prompts.append(None)
    else:
        messages = [
            {
                "role": "system",
                "content": [{"type": "text", "text": SYSTEM_PROMPT}]
            },
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": p}
                ]
            }
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        prompts.append(prompt)

    
# indices & results placeholder
non_empty_indices = [i for i, p in enumerate(prompts) if p is not None]
results = [""] * len(prompts)

# batch loop (simple)
for start in tqdm(range(0, len(non_empty_indices), BATCH_SIZE), desc="Batched generation"):
    batch_indices = non_empty_indices[start:start + BATCH_SIZE]
    batch_prompts = [prompts[i] for i in batch_indices]
    try:
        outs = llm.generate(batch_prompts, sampling_params=sampling)
        for j, out in enumerate(outs):
            text = tokenizer.decode(out.outputs[0].token_ids, skip_special_tokens=True).strip("\n")
            results[batch_indices[j]] = text
    except Exception as e:
        for i in batch_indices:
            results[i] = f"[ERROR] {type(e).__name__}: {str(e)[:200]}"

# save back to dataframe
df["oneshot_response"] = results


df.to_excel(OUTPUT_PATH_1, index=False, engine="openpyxl")

In [ ]:

# Apply to dataframe (no in-place overwrite until all succeed)
df["extracted_zeroshot_response"] = df["zeroshot_response"].apply(lambda s: _parse_loose_json(_extract_last_json_str(s)))
df["extracted_oneshot_response"] = df["oneshot_response"].apply(lambda s: _parse_loose_json(_extract_last_json_str(s)))
df.to_excel(OUTPUT_PATH_2, index=False, engine="openpyxl")